# Batch Processing for Multiple LIF Containers

This notebook processes `.lif` containers in batch mode with no visualization.

Outputs are written to `../results/<lif_container_id>/<lif_image_name>.csv`.

It reuses segmentation and feature-extraction utilities from `../src` and includes researcher-friendly progress/logging with `tqdm`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch

cwd = Path.cwd().resolve()
src_dir = cwd / "src" if (cwd / "src").exists() else cwd.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from utils.batch_processing import build_runtime_config, run_batch

pd.set_option("display.max_columns", None)

In [ ]:
# Paths for input data, results output and UNet3D model loading
RAW_DATA_DIRECTORY = Path("../raw_data")
RESULTS_ROOT = Path("../results")
MODEL_DIR = Path("../plantseg_models/lightsheet_3D_unet_root_ds3x")

# Marker configuration: (marker_name, channel_index, role)
MARKERS = (
    ("edCerulean_CTRL", 0, "DD"),
    ("edCitrine_FRET", 1, "DA"),
    ("edCitrine_CTRL", 2, "AA"),
    ("brightfield", 3, "root_structure"),
)

# Nuclei segmentation settings
NUCLEI_CHANNEL = 2
MIN_MAX_NUCLEI_VOLUME = (250, 4000)

# Root boundary inference settings for UNet3D
INFERENCE_PATCH = (80, 160, 160)
INFERENCE_PATCH_HALO = (8, 16, 16)
INFERENCE_STRIDE_RATIO = 0.75
INFERENCE_BATCH_SIZE = 1
INFERENCE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
INFERENCE_USE_AMP = True

# Root segmentation refinement settings
ROOT_PROBABILITY_THRESHOLD = 0.9
ROOT_OCCUPANCY_THRESHOLD = 0.9
ROOT_FILL_SLICE_AWARE = True
ROOT_SMOOTH_EROSION = 5
ROOT_SMOOTHING = 3
ROOT_WRAP_PERCENTAGE_THRESHOLD = 5.0
ROOT_WRAP_EDT_THRESHOLD = 15.0
DEPTH_PAD_FULL_ROOT = False

# Batch behavior
# Optionally specify which containers and which images within those containers to process.
# - By default, both are set to None, meaning all containers and all images in each container will be processed.
# - These indices are zero-based Python indices (i.e., counting starts from zero).
#   For example, CONTAINER_INDICES = [0, 2, 3] will select the first, third, and fourth containers.
#   Similarly, IMAGE_INDICES = [0, 1, 5] will select the first, second, and sixth images within each selected container.
# - To process only specific containers, set CONTAINER_INDICES to a list of zero-based indices.
# - To process only specific images within each selected container, set IMAGE_INDICES to a list of zero-based indices.
# - You can set only CONTAINER_INDICES, only IMAGE_INDICES, or both, depending on your selection needs.
CONTAINER_INDICES: list[int] | None = None   # e.g. [0, 2, 3]
IMAGE_INDICES: list[int] | None = None       # e.g. [0, 1, 5]
# If True, existing CSV results will be overwritten and all computations (e.g., generating np.arrays such as nuclei_labels, root_3d_mask, etc.) will be performed and saved;
# if False, previously saved CSV results are preserved, and repeated computational steps (such as computing new nuclei_labels and other arrays) are skipped for images that already have results.
OVERWRITE_CSV = False

In [ ]:
NOTEBOOK_CONFIG = {
    "results_root": str(RESULTS_ROOT),
    "model_dir": str(MODEL_DIR),
    "markers": [
        {"name": name, "channel": channel, "role": role}
        for name, channel, role in MARKERS
    ],
    "nuclei_channel": NUCLEI_CHANNEL,
    "min_max_nuclei_volume": list(MIN_MAX_NUCLEI_VOLUME),
    "root_probability_threshold": ROOT_PROBABILITY_THRESHOLD,
    "root_occupancy_threshold": ROOT_OCCUPANCY_THRESHOLD,
    "root_fill_slice_aware": ROOT_FILL_SLICE_AWARE,
    "root_smooth_erosion": ROOT_SMOOTH_EROSION,
    "root_smoothing": ROOT_SMOOTHING,
    "root_wrap_percentage_threshold": ROOT_WRAP_PERCENTAGE_THRESHOLD,
    "root_wrap_edt_threshold": ROOT_WRAP_EDT_THRESHOLD,
    "depth_pad_full_root": DEPTH_PAD_FULL_ROOT,
    "inference_patch": list(INFERENCE_PATCH),
    "inference_patch_halo": list(INFERENCE_PATCH_HALO),
    "inference_stride_ratio": INFERENCE_STRIDE_RATIO,
    "inference_batch_size": INFERENCE_BATCH_SIZE,
    "inference_device": INFERENCE_DEVICE,
    "inference_use_amp": INFERENCE_USE_AMP,
    "container_indices": CONTAINER_INDICES,
    "image_indices": IMAGE_INDICES,
    "overwrite_csv": OVERWRITE_CSV,
}

CONFIG = build_runtime_config(
    user_config=NOTEBOOK_CONFIG,
    raw_data_directory=RAW_DATA_DIRECTORY,
)

In [ ]:
batch_results = run_batch(CONFIG)
batch_results